In [5]:
import gym
import numpy as np

# Initialize the environment
env = gym.make("CartPole-v1")
n_actions = env.action_space.n  # Number of actions
n_states = 10  # Discretize state space for simplicity

# Q-table initialization
q_table = np.zeros((n_states, n_states, n_states, n_states, n_actions))
learning_rate = 0.1
discount_factor = 0.99
epsilon = 1.0  # Exploration rate
epsilon_decay = 0.995
min_epsilon = 0.01
episodes = 5000

# Helper to discretize the state space
def discretize_state(state):
    bins = [
        np.linspace(-2.4, 2.4, n_states - 1),  # Cart position
        np.linspace(-3.0, 3.0, n_states - 1),  # Cart velocity
        np.linspace(-0.5, 0.5, n_states - 1),  # Pole angle
        np.linspace(-2.0, 2.0, n_states - 1),  # Pole angular velocity
    ]
    state_indices = tuple(np.digitize(state[i], bins[i]) for i in range(len(state)))
    return state_indices

# Training loop
for episode in range(episodes):
    state = discretize_state(env.reset()[0])  # Initialize state
    done = False
    total_reward = 0
    
    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()  # Explore
        else:
            action = np.argmax(q_table[state])  # Exploit
        
        # Take action
        next_state_raw, reward, done, _, _ = env.step(action)
        next_state = discretize_state(next_state_raw)
        
        # Update Q-value
        best_next_action = np.argmax(q_table[next_state])
        q_table[state + (action,)] += learning_rate * (
            reward + discount_factor * q_table[next_state + (best_next_action,)] - q_table[state + (action,)]
        )
        
        state = next_state
        total_reward += reward
    
    # Decay epsilon
    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    
    if episode % 500 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}")

print("Training finished.")

# Test the policy
state = discretize_state(env.reset()[0])
done = False
while not done:
    env.render()
    action = np.argmax(q_table[state])
    next_state_raw, _, done, _, _ = env.step(action)
    state = discretize_state(next_state_raw)

env.close()


c:\Users\david\miniconda3\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode: 0, Total Reward: 23.0
Episode: 500, Total Reward: 14.0
Episode: 1000, Total Reward: 15.0
Episode: 1500, Total Reward: 23.0
Episode: 2000, Total Reward: 16.0
Episode: 2500, Total Reward: 14.0
Episode: 3000, Total Reward: 14.0
Episode: 3500, Total Reward: 14.0
Episode: 4000, Total Reward: 15.0
Episode: 4500, Total Reward: 14.0
Training finished.


c:\Users\david\miniconda3\Lib\site-packages\gym\envs\classic_control\cartpole.py:211: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym("CartPole-v1", render_mode="rgb_array")
  gym.logger.warn(
